In [10]:
from google.colab import drive
drive.mount("/content/drive")

import os

# Shared app folder: BOTH notebooks save their files here.
# Keep this path identical in the two notebooks.
project_directory = "/content/drive/MyDrive/Applied Data Science/Project/Optimizing_direct_marketing/Optimizing_direct_marketing/"
app_directory = project_directory + "app/"
os.makedirs(app_directory, exist_ok=True)
os.chdir(app_directory)
print("Working folder:", os.getcwd())

# Folder containing bank-full.csv (same as in Part 3)
input_directory = "/content/drive/MyDrive/Applied Data Science/Python_DataSet/Bank Dataset/"
print("bank-full.csv found:", os.path.exists(input_directory + "bank-full.csv"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder: /content/drive/MyDrive/Applied Data Science/Project/Optimizing_direct_marketing/Optimizing_direct_marketing/app
bank-full.csv found: True


In [11]:
# ======================================================================
# TRAIN AND SAVE THE DEPLOYABLE MODEL
# ----------------------------------------------------------------------
# Reuses from Part-3-machineLearningModels.ipynb (nothing re-tuned or
# re-selected here):
#   - preprocessing (StandardScaler + OneHotEncoder(drop="first"))
#   - the 43 features chosen by VarianceThreshold(0.01)
#   - XGBoost + SMOTE hyperparameters found by GridSearchCV
#   - the 80/20 stratified split (random_state=42)
#
# Differences from Part 3, both to avoid information leakage:
#   1. The averages behind `high_balance` and `long_duration` are computed
#      on the training set only (Part 3 used the full dataset).
#   2. The yes/no cut-off is chosen by 3-fold cross-validation on the
#      training set; the test set is used once, for the final evaluation.
#
# The app scores clients AFTER a first call (to prioritize follow-ups),
# so call duration is known and used as a feature.
# ======================================================================
import json

import joblib
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (average_precision_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

# ----------------------------------------------------------------------
# 0. Settings
# ----------------------------------------------------------------------
DATA_FILE = "bank-full.csv"      # full dataset (45,211 rows), as in the Part 3 results
USE_DURATION_FEATURES = True     # False = pre-call model (drops duration features)
RANDOM_STATE = 42

NUMERIC_FEATURES = ["age", "balance", "campaign", "pdays", "previous", "duration", "day"]
CATEGORICAL_FEATURES = ["job", "month", "marital", "education", "contact", "poutcome"]
PASSTHROUGH_FEATURES = ["default", "housing", "loan", "high_balance", "long_duration", "no_loans"]

# Features selected by VarianceThreshold(0.01) in Part 3 (best XGBoost results).
# prev_success and hypothesis_group are absent: they were 0 for every row in
# Part 3 (poutcome is text, not 3), so the variance threshold removed them.
MODEL_FEATURES = [
    "scaler__age", "scaler__balance", "scaler__campaign", "scaler__pdays", "scaler__previous",
    "scaler__duration", "scaler__day",
    "onehot__job_blue-collar", "onehot__job_entrepreneur", "onehot__job_housemaid",
    "onehot__job_management", "onehot__job_retired", "onehot__job_self-employed",
    "onehot__job_services", "onehot__job_student", "onehot__job_technician", "onehot__job_unemployed",
    "onehot__month_aug", "onehot__month_feb", "onehot__month_jan", "onehot__month_jul",
    "onehot__month_jun", "onehot__month_mar", "onehot__month_may", "onehot__month_nov",
    "onehot__month_oct", "onehot__month_sep",
    "onehot__marital_married", "onehot__marital_single",
    "onehot__education_secondary", "onehot__education_tertiary", "onehot__education_unknown",
    "onehot__contact_telephone", "onehot__contact_unknown",
    "onehot__poutcome_other", "onehot__poutcome_success", "onehot__poutcome_unknown",
    "remainder__default", "remainder__housing", "remainder__loan",
    "remainder__high_balance", "remainder__long_duration", "remainder__no_loans",
]

if not USE_DURATION_FEATURES:
    NUMERIC_FEATURES = [c for c in NUMERIC_FEATURES if c != "duration"]
    PASSTHROUGH_FEATURES = [c for c in PASSTHROUGH_FEATURES if c != "long_duration"]
    MODEL_FEATURES = [f for f in MODEL_FEATURES
                      if f not in ("scaler__duration", "remainder__long_duration")]

INPUT_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES + PASSTHROUGH_FEATURES


def prepare_features(clients, avg_balance, avg_duration):
    """Turn raw client rows (text yes/no values) into the model's input columns.
    The app uses the same logic, with the averages saved in model_info.json."""
    df = clients.copy()
    for col in ["default", "housing", "loan"]:
        df[col] = df[col].map({"yes": 1, "no": 0})
    df["high_balance"] = (df["balance"] > avg_balance).astype(int)
    df["no_loans"] = ((df["loan"] == 0) & (df["housing"] == 0)).astype(int)
    if USE_DURATION_FEATURES:
        df["long_duration"] = (df["duration"] > avg_duration).astype(int)
    return df[INPUT_COLUMNS]


# ----------------------------------------------------------------------
# 1. Load the raw data and split it (same split as Part 3)
# ----------------------------------------------------------------------
raw_df = pd.read_csv(input_directory + DATA_FILE, sep=";")
print(f"Rows loaded: {len(raw_df):,}")
assert len(raw_df) == 45211, "Expected bank-full.csv (45,211 rows); check DATA_FILE"

y = raw_df["y"].map({"yes": 1, "no": 0})
raw_train, raw_test, y_train, y_test = train_test_split(
    raw_df.drop(columns=["y"]), y,
    test_size=0.20, random_state=RANDOM_STATE, stratify=y,
)

# ----------------------------------------------------------------------
# 2. Engineered features, using averages from the TRAINING set only
# ----------------------------------------------------------------------
avg_balance = raw_train["balance"].mean()
avg_duration = raw_train["duration"].mean()

X_train = prepare_features(raw_train, avg_balance, avg_duration)
X_test = prepare_features(raw_test, avg_balance, avg_duration)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,} | Model features: {len(MODEL_FEATURES)}")

# ----------------------------------------------------------------------
# 3. Pipeline: Part 3 preprocessing -> variance-selected features ->
#    SMOTE -> XGBoost with the tuned hyperparameters.
#    Scaler, encoder and SMOTE are fitted on training data only;
#    SMOTE is skipped automatically at prediction time.
# ----------------------------------------------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("scaler", StandardScaler(), NUMERIC_FEATURES),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"),
         CATEGORICAL_FEATURES),
    ],
    remainder="passthrough",
).set_output(transform="pandas")

select_features = ColumnTransformer(
    [("keep", "passthrough", MODEL_FEATURES)],
    remainder="drop",
    verbose_feature_names_out=False,
)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()   # ~7.55, as in Part 3

model = ImbPipeline([
    ("preprocess", preprocessor),
    ("select", select_features),
    ("smote", SMOTE(k_neighbors=5, random_state=RANDOM_STATE)),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
    )),
])

# ----------------------------------------------------------------------
# 4. Choose the yes/no cut-off on the TRAINING set (3-fold CV, best F1)
#    Each training client is scored by a model that did not see it.
# ----------------------------------------------------------------------
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_proba = cross_val_predict(model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

candidates = np.round(np.arange(0.05, 0.96, 0.01), 2)
cv_f1 = [f1_score(y_train, cv_proba >= t) for t in candidates]
DECISION_THRESHOLD = float(candidates[int(np.argmax(cv_f1))])
print(f"Decision threshold (best cross-validated F1): {DECISION_THRESHOLD}")

# ----------------------------------------------------------------------
# 5. Train on the full training set and evaluate ONCE on the test set
# ----------------------------------------------------------------------
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
y_true = y_test.to_numpy()
y_pred = (proba >= DECISION_THRESHOLD).astype(int)

metrics = {
    "roc_auc": roc_auc_score(y_true, proba),
    "pr_auc": average_precision_score(y_true, proba),
    "f1": f1_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred),
    "recall": recall_score(y_true, y_pred),
}
print()
for name, value in metrics.items():
    print(f"{name:<10} {value:.4f}")
print("\nConfusion matrix:\n", confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=["No", "Yes"]))

# Precision / recall at every cut-off, displayed in the app's cut-off tab
# (reporting only; not used to choose anything)
thresholds = []
for t in candidates:
    pred = proba >= t
    flagged = int(pred.sum())
    hits = int((pred & (y_true == 1)).sum())
    thresholds.append({
        "threshold": float(t),
        "precision": hits / flagged if flagged else None,
        "recall": hits / int(y_true.sum()),
        "share_called": flagged / len(y_true),
    })

# ----------------------------------------------------------------------
# 6. Save the model and everything the app needs
# ----------------------------------------------------------------------
joblib.dump(model, "model.joblib")

model_info = {
    **{name: round(float(value), 4) for name, value in metrics.items()},
    "base_rate": round(float(y_train.mean()), 4),
    "decision_threshold": DECISION_THRESHOLD,
    "uses_duration": USE_DURATION_FEATURES,
    "avg_balance": float(avg_balance),
    "avg_duration": float(avg_duration),
    "input_columns": INPUT_COLUMNS,
    "model_features": MODEL_FEATURES,
    "options": {c: sorted(raw_df[c].unique().tolist()) for c in CATEGORICAL_FEATURES},
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "thresholds": thresholds,
}
with open("model_info.json", "w") as f:
    json.dump(model_info, f, indent=2)

# Unseen (test) clients in the raw format a user would upload
raw_test.sample(300, random_state=7).to_csv("sample_clients.csv", index=False)

# ----------------------------------------------------------------------
# 7. Reload check: score raw test clients exactly as the app will
# ----------------------------------------------------------------------
reloaded = joblib.load("model.joblib")
with open("model_info.json") as f:
    saved_info = json.load(f)
check = reloaded.predict_proba(
    prepare_features(raw_test.head(20), saved_info["avg_balance"], saved_info["avg_duration"])
)[:, 1]
assert np.allclose(check, proba[:20]), "Reloaded model gives different predictions"
print("Reload check passed. Saved: model.joblib, model_info.json, sample_clients.csv")

Rows loaded: 45,211
Train: 36,168 | Test: 9,043 | Model features: 43
Decision threshold (best cross-validated F1): 0.7

roc_auc    0.9282
pr_auc     0.6050
f1         0.6148
precision  0.5090
recall     0.7760

Confusion matrix:
 [[7193  792]
 [ 237  821]]
              precision    recall  f1-score   support

          No       0.97      0.90      0.93      7985
         Yes       0.51      0.78      0.61      1058

    accuracy                           0.89      9043
   macro avg       0.74      0.84      0.77      9043
weighted avg       0.91      0.89      0.90      9043

Reload check passed. Saved: model.joblib, model_info.json, sample_clients.csv
